## Cohort alluvial

Three-axis alluvial ([ggalluvial](https://corybrunson.github.io/ggalluvial/)) of the patient cohort: `cancer_type` -> molecular subclass -> `amplicon_class`. Ribbons are colored by amplicon class and carry that color across all three axes, so you can trace which molecular subclasses within a tumor type carry ecDNA / intrachromosomal amplifications.

Notes:
- The plot is wrapped in `plot_alluvial()`, with the `axes` (any number, >= 2) and the interior-axis label threshold `n` exposed as arguments. Ribbon color follows the last axis by default (`fill_by`).
- `aes.bind = "alluvia"` orders lodes within every stratum by fill, so all ecDNA (then intrachromosomal) group to the top of each tumor type rather than splitting into a bar per subclass.
- The middle axis disambiguates `cancer_subclass` per tumor type (survival's `molecular_subtype` convention, e.g. `LGG: MAPK`); otherwise generic subclasses like `NOS` pool across 14 tumor types into one misleading stratum. NA subclasses become a `<type> NOS` stratum. Interior labels drop the tumor-type prefix (so a label reads `NOS (163)`), but the strata stay type-prefixed internally, so `LGG NOS` and `EPN NOS` remain distinct flows. Only interior strata with at least `n` patients are labeled; the two outer axes are always labeled.
- Right-axis (amplicon) labels are rotated 90 degrees to use vertical space.
- Every patient passes through exactly one stratum per axis, so all three columns sum to the full cohort (n=2967).
- Text is 7 pt Arial throughout (`geom_text` sizes use `7/.pt`, since geom text size is in mm). Amplicon-class colors match the survival KM palette (`survival-plots.R`): ecDNA red, intrachromosomal magenta; `no amplification` kept light grey as a receding background.

In [ ]:
Sys.setenv(LANGUAGE = "en")
suppressMessages({
    library(tidyverse)
    library(readxl)
    library(ggalluvial)
    library(ggrepel)
})

import <- new.env()
source("../../src/data_imports.R", local = import)
plotting <- new.env()
source("../../src/plotting.R",local=plotting)

In [ ]:
H=8;W=6
options(repr.plot.width = W, repr.plot.height = H)

path = "../../data/Supplementary\ Tables.xlsx"
patients <- import$patients(path)

# axis level orders (top-level so plot calls can pass amp_levels as fill_breaks)
amp_levels <- c("ecDNA", "intrachromosomal", "no amplification")  # ecDNA first

# amplicon-class palette matches survival KM colors (survival-plots.R km_palette);
# no amplification kept light grey as a receding background.
amp_colors <- c(
    "ecDNA"            = "red",
    "intrachromosomal" = "magenta",
    "no amplification" = "grey85"
)

# global text defaults: 7 pt, Arial. geom_text size is in mm, so convert for a true 7 pt as `FS/.pt`.
FS <- 7; FF <- "Arial"

format_patient_table <- function(patients){
    # tumor types ordered by frequency (LGG first -> top)
    ct_levels <- patients %>% count(cancer_type, sort = TRUE) %>% pull(cancer_type)

    df <- patients %>%
        filter(!is.na(cancer_type)) %>%
        # disambiguate molecular subclass per tumor type (survival's molecular_subtype convention);
        # NOS etc. are otherwise pooled across 14 tumor types.
        mutate(subclass = if_else(is.na(cancer_subclass),
                                  paste0(cancer_type, " NOS"),
                                  paste0(cancer_type, ": ", cancer_subclass))) %>%
        mutate(cancer_type = factor(cancer_type, levels = ct_levels))   # LGG top, rare bottom

    # order subclass strata grouped by tumor type (same top->bottom order), then by size
    sub_levels <- df %>% count(cancer_type, subclass) %>%
        arrange(cancer_type, desc(n)) %>% pull(subclass)
    df %>%
        mutate(subclass       = factor(subclass, levels = sub_levels),
               amplicon_class = factor(amplicon_class, levels = amp_levels))  # ecDNA top
}

df <- format_patient_table(patients)

In [ ]:
plot_alluvial <- function(df,
                          axes        = c("cancer_type", "subclass", "amplicon_class"),
                          axis_labels = gsub("_", " ", axes),
                          fill_by     = tail(axes, 1),
                          fill_colors = NULL,
                          fill_breaks = NULL,
                          n           = 20,   # min stratum size to label an *interior* axis
                          repel_below = 60,   # first-axis strata smaller than this are repelled (leader lines);
                                              # larger ones get a plain label hugging their box
                          title = sprintf("Pediatric pan-cancer cohort (n=%d patients)", nrow(df))) {
    stopifnot(length(axes) >= 2, length(axis_labels) == length(axes))
    k <- length(axes)

    # axis1=..., axis2=..., ... mapping, plus fill carried across the whole path
    axis_map <- rlang::set_names(lapply(axes, rlang::sym), paste0("axis", seq_len(k)))
    mapping  <- aes(!!!axis_map, fill = .data[[fill_by]])

    # per-stratum patient counts. Assumes stratum values are disjoint across axes
    # (true here: cancer_type / "type: subclass" / amplicon_class).
    counts <- bind_rows(lapply(seq_len(k), function(i)
        df %>% count(stratum = as.character(.data[[axes[i]]]), name = "cnt") %>% mutate(idx = i)))

    # interior (subclass) labels drop the leading "<tumor type>: " / "<tumor type> " so they
    # don't repeat the tumor type; strata identity stays type-prefixed, so e.g. LGG NOS and
    # EPN NOS remain distinct flows even though both are labeled "NOS".
    ct_vals   <- as.character(unique(df[[axes[1]]]))
    strip_re  <- paste0("^(", paste(ct_vals[order(-nchar(ct_vals))], collapse = "|"), ")(: | )")
    full_lab  <- setNames(paste0(counts$stratum, " (", counts$cnt, ")"), counts$stratum)
    strip_lab <- setNames(paste0(sub(strip_re, "", counts$stratum), " (", counts$cnt, ")"), counts$stratum)

    # first axis split by size: big strata get a plain label at the box; small ones are repelled.
    # they sit in separate x-columns (big hug the box, small fan further left) so the two label
    # layers -- which don't share repel state -- don't collide.
    first_big   <- counts %>% filter(idx == 1, cnt >= repel_below) %>% pull(stratum)
    first_small <- counts %>% filter(idx == 1, cnt <  repel_below) %>% pull(stratum)
    last_vals   <- as.character(unique(df[[axes[k]]]))
    mid_vals    <- counts %>% filter(idx > 1, idx < k, cnt >= n) %>% pull(stratum)

    lab <- function(vals, lookup) function(s) {
        s <- as.character(s); out <- unname(lookup[s]); out[!(s %in% vals)] <- NA; out
    }

    ggplot(df, mapping) +
        # aes.bind = "alluvia": order lodes within every stratum by fill first (over all other
        # axes), so all ecDNA groups to the top of each tumor type, not per-subclass bands
        geom_alluvium(width = 1/5, alpha = 0.7, aes.bind = "alluvia") +
        geom_stratum(width = 1/5, fill = "grey95", color = "grey40", linewidth = 0.15) +
        # first axis, large strata: plain label hugging the box (no leader needed)
        geom_text(stat = StatStratum, size = FS/.pt, family = FF, na.rm = TRUE, hjust=0.5,
            aes(label = after_stat(lab(first_big, full_lab)(stratum)))) +
        # first axis, small strata: repel further left with leader lines
        ggrepel::geom_text_repel(stat = StatStratum, size = FS/.pt, family = FF, hjust = 1, direction = "y",
            nudge_x = -0.2, segment.size = 0.2, segment.color = "grey70",
            min.segment.length = 0, max.overlaps = Inf, na.rm = TRUE,
            aes(label = after_stat(lab(first_small, full_lab)(stratum)))) +
        # interior axes: label strata >= n (prefix stripped)
        geom_text(stat = StatStratum, size = FS/.pt, family = FF, na.rm = TRUE,
            aes(label = after_stat(lab(mid_vals, strip_lab)(stratum)))) +
        # last axis (right): rotate 90 so long labels use vertical space instead of running off-canvas
        ggrepel::geom_text_repel(stat = StatStratum, size = FS/.pt, family = FF, angle = 90, hjust = 0.5, vjust = 0.5,
            nudge_x = 0.2, na.rm = TRUE, segment.size = 0.2, segment.color = "grey70",
            aes(label = after_stat(lab(last_vals, full_lab)(stratum)))) +
        scale_fill_manual(values = fill_colors, breaks = fill_breaks, name = gsub("_", " ", fill_by)) +
        scale_x_continuous(breaks = seq_len(k), labels = axis_labels,
                           expand = expansion(mult = c(0.12, 0.05))) +
        labs(title = title, y = "patients") +
        theme_minimal(base_size = FS, base_family = FF) +
        theme(text = element_text(size = FS, family = FF),
              plot.title = element_text(size = FS), axis.title = element_text(size = FS),
              axis.text = element_text(size = FS), legend.title = element_text(size = FS),
              legend.text = element_text(size = FS),
              panel.grid = element_blank(), axis.text.y = element_blank(),
              axis.title.x = element_blank(), legend.position = "bottom")
}

In [ ]:
p <- plot_alluvial(df,
    axes = c("cancer_type", "subclass", "amplicon_class"),
    axis_labels = c("tumor type", "molecular subclass", "amplicon class"),
    fill_colors = amp_colors, fill_breaks = amp_levels,
    n = 50, repel_below=40)
show(p)
plotting$save_ggplot("alluvial_patients", plot=p, width = W, height = H, units='in')

In [ ]:
p <- plot_alluvial(df,
    axes = c("cancer_type", "amplicon_class"),
    axis_labels = c("tumor type", "amplicon class"),
    fill_colors = amp_colors, fill_breaks = amp_levels,
)
show(p)